In [1]:
import json
from pathlib import Path

# Just a simple test all is working as expected
analysis_dir = Path.cwd() if (Path.cwd().parent / 'openapi_specification.json').exists() else Path.cwd() / 'analysis'
spec_path = analysis_dir.parent / 'openapi_specification.json'
spec = json.loads(spec_path.read_text(encoding='utf-8'))
pretty_spec_path = analysis_dir / 'openapi_specification_pretty.json'
pretty_spec_path.write_text(json.dumps(spec, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print(f'Formatted specification: {pretty_spec_path.resolve()}')
print(json.dumps(spec['paths']['/api/market']['get'], indent=2))

Formatted specification: C:\projects\eve_online_streamer\analysis\openapi_specification_pretty.json
{
  "tags": [
    "market"
  ],
  "summary": "Get Market Data",
  "description": "Get Online market data with price drop opportunities\n\nReturns market data for items where the price has dropped below a specified threshold\ncompared to the average price over the specified timeframe.",
  "operationId": "get_market_data_api_market_get",
  "parameters": [
    {
      "name": "region_id",
      "in": "query",
      "required": false,
      "schema": {
        "anyOf": [
          {
            "type": "integer"
          },
          {
            "type": "null"
          }
        ],
        "description": "Online region ID",
        "title": "Region Id"
      },
      "description": "Online region ID"
    },
    {
      "name": "item_id",
      "in": "query",
      "required": false,
      "schema": {
        "anyOf": [
          {
            "type": "integer"
          },
          {
  

In [4]:
# Threshold consistency: sample identical query settings at several thresholds.
"""
Swagger showed that the threshold is unreliable, this confirms it.
"""
import os
import requests

market_url = os.environ.get('MARKET_API_BASE_URL', 'http://localhost:8000').rstrip('/') + '/api/market'
threshold_samples = {}
for threshold in (10, 30, 50):
    response = requests.get(market_url, params={'threshold': threshold, 'timeframe': 'weekly', 'per_page': 20}, timeout=10)
    response.raise_for_status()
    threshold_samples[threshold] = response.json()
    print(f"For threshold{threshold} we have N= {len(threshold_samples[threshold]['data'])} items", 'meta:', threshold_samples[threshold]['meta'])
    diffs = [
    -item['price_change'] - threshold
    for item in threshold_samples[threshold]['data']
    ]
    
    correct = sum(diff >= 0 for diff in diffs)
    incorrect = sum(diff < 0 for diff in diffs)
    
    print(f"Threshold: {threshold}%")
    print(f"Correct: {correct}, Incorrect: {incorrect}")
    print(f"Difference from threshold (percentage points): {[round(d, 1) for d in diffs]}")


For threshold10 we have N= 20 items meta: {'region_id': 10000043, 'region_name': 'Domain', 'threshold': 10.0, 'timeframe': 'weekly', 'timestamp': '2026-10-14T10:11:12.513269'}
Threshold: 10%
Correct: 6, Incorrect: 14
Difference from threshold (percentage points): [13.7, 12.7, 9.0, 1.4, 0.2, 0.1, -1.7, -4.0, -6.8, -12.6, -13.4, -14.0, -18.7, -22.3, -23.8, -24.0, -25.9, -30.3, -30.8, -33.9]
For threshold30 we have N= 20 items meta: {'region_id': 10000020, 'region_name': 'Tash-Murkon', 'threshold': 30.0, 'timeframe': 'weekly', 'timestamp': '2026-10-14T10:11:12.680006'}
Threshold: 30%
Correct: 1, Incorrect: 19
Difference from threshold (percentage points): [10.5, -7.6, -8.8, -13.9, -14.2, -15.2, -19.9, -23.4, -27.9, -32.6, -35.8, -39.2, -40.3, -44.2, -44.4, -46.0, -46.8, -50.1, -51.8, -52.3]
For threshold50 we have N= 20 items meta: {'region_id': 10000043, 'region_name': 'Domain', 'threshold': 50.0, 'timeframe': 'weekly', 'timestamp': '2026-10-14T10:11:12.779625'}
Threshold: 50%
Correct: 2

In [6]:
import requests
import time
from datetime import datetime, timezone

url = "http://localhost:8000/api/market"

for run in range(3):
    fetched_at = datetime.now(timezone.utc)

    response = requests.get(
        url,
        params={"per_page": 100, "page": 1},
        timeout=10,
    )
    response.raise_for_status()

    items = response.json()["data"]
    timestamps = [
        datetime.fromisoformat(item["timestamp"])
        for item in items
    ]

    print(f"\nRun {run + 1}")
    print(f"Fetched at:       {fetched_at.isoformat()}")
    print(f"Earliest event:   {min(timestamps)}")
    print(f"Latest event:     {max(timestamps)}")
    print(f"Unique timestamps:{len(set(timestamps))}")
    print(f"Sorted ascending: {timestamps == sorted(timestamps)}")
    print(f"Sorted descending:{timestamps == sorted(timestamps, reverse=True)}")

    time.sleep(5)


Run 1
Fetched at:       2026-10-14T10:17:42.401651+00:00
Earliest event:   2026-10-14 09:46:50.524689
Latest event:     2026-10-14 10:17:16.525095
Unique timestamps:100
Sorted ascending: False
Sorted descending:False

Run 2
Fetched at:       2026-10-14T10:17:47.528597+00:00
Earliest event:   2026-10-14 09:47:18.644917
Latest event:     2026-10-14 10:16:52.645483
Unique timestamps:100
Sorted ascending: False
Sorted descending:False

Run 3
Fetched at:       2026-10-14T10:17:52.648514+00:00
Earliest event:   2026-10-14 09:46:53.721086
Latest event:     2026-10-14 10:17:13.720539
Unique timestamps:100
Sorted ascending: False
Sorted descending:False


In [12]:
# Pagination predictability: fetch page 1 twice, with page 2 in between.
"""
We see that the item-id's are consistent
"""

pages = []
for page in (1, 2, 1):
    response = requests.get(market_url, params={'threshold': 30, 'timeframe': 'weekly', 'page': page, 'per_page': 5}, timeout=10)
    response.raise_for_status()
    pages.append(response.json())
for requested_page, body in zip((1, 2, 1), pages):
    print('requested page:', requested_page, 'pagination:', body['pagination'])
    print('item IDs:', [item['item_id'] for item in body['data']])


requested page: 1 pagination: {'current_page': 1, 'per_page': 5, 'total_pages': 1000, 'total_items': 5000}
item IDs: [35, 38, 37, 34, 36]
requested page: 2 pagination: {'current_page': 2, 'per_page': 5, 'total_pages': 1000, 'total_items': 5000}
item IDs: [39, 40, 17888, 16272, 16274]
requested page: 1 pagination: {'current_page': 1, 'per_page': 5, 'total_pages': 1000, 'total_items': 5000}
item IDs: [38, 34, 37, 35, 36]


In [13]:
# Compare the two page 1 responses by item ID.
"""
The rest of the content is not consistent, though
"""
first = {item["item_id"]: item for item in pages[0]["data"]}
second = {item["item_id"]: item for item in pages[2]["data"]}

for item_id in sorted(first.keys() & second.keys()):
    a = first[item_id]
    b = second[item_id]

    print(
        f"{a['item_name']:12} "
        f"price: {a['current_price']:8.2f} -> {b['current_price']:8.2f} "
        f"timestamp: {a['timestamp']} -> {b['timestamp']}"
    )

Tritanium    price:     7.95 ->     6.97 timestamp: 2026-10-14T10:15:22.646057 -> 2026-10-14T09:58:43.895313
Pyerite      price:     5.42 ->     8.67 timestamp: 2026-10-14T10:21:19.646086 -> 2026-10-14T09:59:26.895342
Mexallon     price:    72.83 ->    71.08 timestamp: 2026-10-14T10:04:53.646096 -> 2026-10-14T10:20:09.895351
Isogen       price:   123.40 ->   117.17 timestamp: 2026-10-14T10:20:28.646105 -> 2026-10-14T10:02:18.895359
Nocxium      price:   294.68 ->   347.04 timestamp: 2026-10-14T10:01:02.646112 -> 2026-10-14T10:02:37.895367


In [14]:
# Checking whether the region and station are consistent
"""
Region: yes, station: no
"""
for item_id in sorted(first.keys() & second.keys()):
    a = first[item_id]
    b = second[item_id]

    print(
        f"{a['item_name']:12} "
        f"region: {a['location']['region_id']} -> {b['location']['region_id']} | "
        f"station: {a['location']['station_id']} -> {b['location']['station_id']}"
    )

Tritanium    region: 10000002 -> 10000002 | station: 60003760 -> 60003760
Pyerite      region: 10000043 -> 10000043 | station: 60004588 -> 60004588
Mexallon     region: 10000030 -> 10000030 | station: 60008494 -> 60008494
Isogen       region: 10000032 -> 10000032 | station: 60011866 -> 60011866
Nocxium      region: 10000042 -> 10000042 | station: 60008494 -> 60003760
